# Génération des réseaux de formes à partir des règles
- Évaluation  

## Importations
- codecs pour les encodages
- pandas et numpy pour les calculs sur tableaux
- matplotlib pour les graphiques
- itertools pour les itérateurs sophistiqués (paires sur liste, ...)

In [76]:
# -*- coding: utf8 -*-
import codecs,operator,datetime,os,glob
import features
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools as it
import pickle
import networkx as nx
#%pylab inline
#pd.options.display.mpl_style = 'default'
debug=False
from __future__ import print_function

def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

In [77]:
%matplotlib inline

In [78]:
import yaml

In [79]:
from IPython.display import display, HTML

In [80]:
import datetime
def dateheure():
    return datetime.datetime.utcnow().strftime('%y%m%d%H%M')

In [81]:
saut="\n"

In [82]:
features.add_config('/Users/gilles/Github/SWIM/ParadigmGeneration/Vlexique2/bdlexique.ini')
fs=features.FeatureSystem('phonemes')

# Choix de l'échantillon et du gold
- *sampleFile* est le nom de l'échantillon de départ
- *goldFile* est le nom du lexique Gold de référence

In [83]:
repFiles="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
inputFile="vlexique2-S8.csv"
outputFile="vlexique2-S8-omp-Swim2.csv"
goldFile="vlexique2-R8.csv"
platinumFile="vlexique2-Total.csv"
fInput=repFiles+inputFile
fOutput=repFiles+outputFile
fGold=repFiles+goldFile
fPlatinum=repFiles+platinumFile

In [84]:
phonologicalMap="-X"
if "omp" in outputFile:
    casesType="-Morphomes"
else:
    casesType=""
listeFormesOutput=["FS","FP"]

### Dédoubler les lignes avec des surabondances dans *colonne*
>identifier une ligne avec surabondance

>>ajouter les lignes correspondant à chaque valeur

>>ajouter le numéro de la ligne initiale dans les lignes à supprimer

>supprimer les lignes avec surabondance

NB : il faut préparer le tableau pour avoir une indexation qui permette l'ajout des valeurs individuelles et la suppression des lignes de surabondances

In [85]:
def splitCellMates(df,colonne):
    '''
    Calcul d'une dataframe sans surabondance par dédoublement des valeurs
    '''
    test=df.reset_index()
    del test["index"]
    splitIndexes=[]
    for index,ligne in test.iterrows():
        if "," in ligne[colonne]:
            valeurs=set(ligne[colonne].split(","))
            nouvelleLigne=ligne
            for valeur in valeurs:
                nouvelleLigne[colonne]=valeur
                test=test.append(nouvelleLigne,ignore_index=True)
            splitIndexes.append(index)
    if splitIndexes:
        test=test.drop(test.index[splitIndexes])
    return test


# Lecture de l'échantillon

In [86]:
neutralisationsNORD=(u"6û",u"9ê")
neutralisationsSUD=(u"e2o",u"E9O")
if phonologicalMap=="-N":
    neutralisations=neutralisationsNORD
elif phonologicalMap=="-S":
    neutralisations=neutralisationsSUD
else:
    neutralisations=(u"",u"")
    phonologicalMap=("-X")
bdlexiqueIn = u"èò"+neutralisations[0]
bdlexiqueNum = [ord(char) for char in bdlexiqueIn]
neutreOut = u"EO"+neutralisations[1]
neutralise = dict(zip(bdlexiqueNum, neutreOut))

In [87]:
def recoder(chaine,table=neutralise):
    if type(chaine)==str:
        temp=chaine.translate(table)
        result=temp
    elif type(chaine)==unicode:
        result=chaine.translate(table)
    else:
        result=chaine
    return result

### Vérification de la phonotactique des glides du français
- si *prononciation* est *None* renvoyer *None*
- ajout de diérèses dans les séquences mal-formées
- vérification des séquences consonne+glide à la finale

In [88]:
dierese={"j":"ij", "w":"uw","H":"yH","i":"ij","u":"uw","y":"yH"}

In [89]:
def checkFrench(prononciation):
    if prononciation and not pd.isnull(prononciation):
        result=recoder(prononciation)
        m=re.match(r"^.*([^ieèEaOouy926êôâ])[jwH]$",result)
        if m:
            print ("pb avec un glide final", [prononciation])
        m=re.match(r"(.*[ptkbdgfsSvzZ][rl])([jwH])(.*)",result)
        if m:
            n=re.search(r"[ptkbdgfsSvzZ][rl](wa|Hi|wê)",result)
            if not n:
                glide=m.group(2)
                result=m.group(1)+dierese[glide]+m.group(3)
        m=re.match(r"(.*)([iuy])([ieEaOouy].*)",result)
        if m:
            glide=m.group(2)
            result=m.group(1)+dierese[glide]+m.group(3)
    else:
        result=prononciation
    return result

In [90]:
echantillon=pd.read_csv(fInput,sep=";",encoding="utf8")
if u"Unnamed: 0" in echantillon.columns:
    del echantillon[u"Unnamed: 0"]
echantillon=echantillon.dropna(axis=1,how='all')
print(len(echantillon.columns))
echantillon.head()

45


,lexeme,ai1S,ai2P,ai3P,ai3S,fi1P,fi1S,fi2P,fi2S,fi3P,...,ppFP,ppFS,ppMP,ppMS,ps1P,ps1S,ps2P,ps2S,ps3P,ps3S
0,abaisser,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,abandonner,NaN,NaN,NaN,NaN,NaN,abâdOn6rE,NaN,NaN,NaN,...,abâdOne,abâdOne,abâdOne,abâdOne,NaN,abâdOn,NaN,NaN,NaN,abâdOn
2,abattre,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,abaty,abaty,abaty,NaN,NaN,NaN,NaN,NaN,NaN
3,abolir,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,aborder,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,abOrde,NaN,NaN,NaN,NaN,NaN,NaN


In [91]:
paradigmes=pd.read_csv(fOutput,sep=";",encoding="utf8")
if u"Unnamed: 0" in paradigmes.columns:
    del paradigmes[u"Unnamed: 0"]
paradigmes=paradigmes.dropna(axis=1,how='all')
print((paradigmes.columns))
paradigmes.head()

Index(['ai1S', 'ai2P', 'ai3P', 'ai3S', 'fi1P', 'fi2P', 'fi2S', 'fi3P', 'fi3S',
       'ii1P', 'ii1S', 'ii2P', 'ii2S', 'ii3P', 'ii3S', 'inf', 'is3S', 'lexeme',
       'pI1P', 'pI2P', 'pI2S', 'pP', 'pc1P', 'pc1S', 'pc2P', 'pc2S', 'pc3P',
       'pc3S', 'pi1P', 'pi1S', 'pi2P', 'pi2S', 'pi3P', 'pi3S', 'ppFP', 'ppFS',
       'ppMP', 'ppMS', 'ps1P', 'ps1S', 'ps2P', 'ps2S', 'ps3P', 'ps3S'],
      dtype='object')


,ai1S,ai2P,ai3P,ai3S,fi1P,fi2P,fi2S,fi3P,fi3S,ii1P,...,ppFP,ppFS,ppMP,ppMS,ps1P,ps1S,ps2P,ps2S,ps3P,ps3S
0,NaN,NaN,abâdOnEr,abâdOna,abâdOn6rô,abâdOn6re,abâdOn6ra,abâdOn6rô,abâdOn6ra,abâdOnjô,...,abâdOne,abâdOne,abâdOne,abâdOne,abâdOnjô,abâdOn,abâdOnje,abâdOn,abâdOn,abâdOn
1,NaN,NaN,NaN,NaN,abatrô,NaN,abatra,abatrô,abatra,NaN,...,abaty,abaty,abaty,abaty,NaN,abat,NaN,abat,abat,abat
2,NaN,NaN,abOrdEr,abOrda,abOrd6rô,abOrd6re,abOrd6ra,abOrd6rô,abOrd6ra,abOrdjô,...,abOrde,abOrde,abOrde,abOrde,abOrdjô,abOrd,abOrdje,abOrd,abOrd,abOrd
3,NaN,NaN,NaN,NaN,abutirô,abutire,abutira,abutirô,abutira,NaN,...,NaN,NaN,abuti,abuti,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,abwaja,abErô,abEre,abEra,abErô,abEra,NaN,...,abwaje,abwaje,abwaje,abwaje,NaN,abwa,NaN,abwa,abwa,abwa


In [92]:
sampleCases=paradigmes.columns.values.tolist()
sampleCases.remove(u"lexeme")
# sampleCases
analyseCases=sampleCases

#Adapt all the forms to French phonology
for case in sampleCases:
    paradigmes[case]=paradigmes[case].apply(lambda x: checkFrench(x))

pb avec un glide final ['kOpj']
pb avec un glide final ['dEdj']
pb avec un glide final ['EkspEdj']
pb avec un glide final ['ymilj']
pb avec un glide final ['êsâdj']
pb avec un glide final ['fOtOgrafj']
pb avec un glide final ['pyrifj']
pb avec un glide final ['r6nj']
pb avec un glide final ['rEkôsilj']
pb avec un glide final ['rEfyZj']
pb avec un glide final ['ymilj']
pb avec un glide final ['fOtOgrafj']
pb avec un glide final ['pyrifj']
pb avec un glide final ['rEkôsilj']
pb avec un glide final ['ymilj']
pb avec un glide final ['fOtOgrafj']
pb avec un glide final ['pyrifj']
pb avec un glide final ['rEkôsilj']
pb avec un glide final ['ymilj']
pb avec un glide final ['fOtOgrafj']
pb avec un glide final ['pyrifj']
pb avec un glide final ['r6nj']
pb avec un glide final ['rEkôsilj']
pb avec un glide final ['rEfyZj']
pb avec un glide final ['ymilj']
pb avec un glide final ['fOtOgrafj']
pb avec un glide final ['pyrifj']
pb avec un glide final ['rEkôsilj']
pb avec un glide final ['sErtifj']
p

- sampleCases pour la liste des cases effectivement représentées dans le corpus de départ 

In [93]:
countInput=paradigmes.stack().value_counts(dropna=True).sum()
print("nombre de formes de départ",countInput)

nombre de formes de départ 48059


In [94]:
gold=pd.read_csv(fGold,sep=";",encoding="utf8")
if u"Unnamed: 0" in gold.columns:
    del gold[u"Unnamed: 0"]
gold=gold.dropna(axis=1,how='all')

for case in sampleCases:
    gold[case]=gold[case].apply(lambda x: checkFrench(x))

gold.head()

,lexeme,ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1P,fi1S,fi2P,...,ppFP,ppFS,ppMP,ppMS,ps1P,ps1S,ps2P,ps2S,ps3P,ps3S
0,abaisser,NaN,abEsE,abEsat,NaN,abEsEr,abEsa,abEs6rô,abEs6rE,abEs6re,...,abEse,abEse,abEse,abEse,abEsjô,abEs,NaN,abEs,abEs,abEs
1,abandonner,abâdOnam,abâdOnE,abâdOnat,abâdOna,abâdOnEr,abâdOna,abâdOn6rô,NaN,abâdOn6re,...,NaN,NaN,NaN,NaN,abâdOnjô,NaN,abâdOnje,abâdOn,abâdOn,NaN
2,abasourdir,NaN,NaN,NaN,abazurdi,NaN,NaN,NaN,abazurdirE,NaN,...,abazurdi,abazurdi,abazurdi,abazurdi,NaN,NaN,NaN,NaN,NaN,abazurdis
3,abattre,NaN,NaN,NaN,abati,abatir,abati,abatrô,abatrE,abatre,...,abaty,NaN,NaN,NaN,abatjô,abat,abatje,abat,abat,abat
4,abdiquer,NaN,NaN,NaN,NaN,NaN,abdika,abdik6rô,abdik6rE,abdik6re,...,NaN,NaN,abdike,abdike,NaN,abdik,NaN,abdik,NaN,abdik


In [95]:
platinum=pd.read_csv(fPlatinum,sep=";",encoding="utf8")
if u"Unnamed: 0" in gold.columns:
    del platinum[u"Unnamed: 0"]
platinum=platinum.dropna(axis=1,how='all')

for case in sampleCases:
    platinum[case]=platinum[case].apply(lambda x: checkFrench(x))

platinum.head()

,lexeme,ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1P,fi1S,fi2P,...,ppFP,ppFS,ppMP,ppMS,ps1P,ps1S,ps2P,ps2S,ps3P,ps3S
0,abaisser,abEsam,abEsE,abEsat,abEsa,abEsEr,abEsa,abEs6rô,abEs6rE,abEs6re,...,abEse,abEse,abEse,abEse,abEsjô,abEs,abEsje,abEs,abEs,abEs
1,abandonner,abâdOnam,abâdOnE,abâdOnat,abâdOna,abâdOnEr,abâdOna,abâdOn6rô,abâdOn6rE,abâdOn6re,...,abâdOne,abâdOne,abâdOne,abâdOne,abâdOnjô,abâdOn,abâdOnje,abâdOn,abâdOn,abâdOn
2,abasourdir,abazurdim,abazurdi,abazurdit,abazurdi,abazurdir,abazurdi,abazurdirô,abazurdirE,abazurdire,...,abazurdi,abazurdi,abazurdi,abazurdi,abazurdisjô,abazurdis,abazurdisje,abazurdis,abazurdis,abazurdis
3,abattre,abatim,abati,abatit,abati,abatir,abati,abatrô,abatrE,abatre,...,abaty,abaty,abaty,abaty,abatjô,abat,abatje,abat,abat,abat
4,abdiquer,abdikam,abdikE,abdikat,abdika,abdikEr,abdika,abdik6rô,abdik6rE,abdik6re,...,abdike,abdike,abdike,abdike,abdikjô,abdik,abdikje,abdik,abdik,abdik


In [96]:
countLexemes=len(paradigmes.dropna(thresh=1)["lexeme"])

In [97]:
paradigmes.loc[paradigmes.lexeme.isin(["abandonner"]),:].T.dropna().to_dict()

{0: {'ai3P': 'abâdOnEr',
  'ai3S': 'abâdOna',
  'fi1P': 'abâdOn6rô',
  'fi2P': 'abâdOn6re',
  'fi2S': 'abâdOn6ra',
  'fi3P': 'abâdOn6rô',
  'fi3S': 'abâdOn6ra',
  'ii1P': 'abâdOnjô',
  'ii1S': 'abâdOnE',
  'ii2P': 'abâdOnje',
  'ii2S': 'abâdOnE',
  'ii3P': 'abâdOnE',
  'ii3S': 'abâdOnE',
  'inf': 'abâdOne',
  'lexeme': 'abandonner',
  'pI1P': 'abâdOnô',
  'pI2P': 'abâdOne',
  'pI2S': 'abâdOn',
  'pP': 'abâdOnâ',
  'pc1S': 'abâdOn6rE',
  'pc2P': 'abâdOn6rje',
  'pc2S': 'abâdOn6rE',
  'pc3P': 'abâdOn6rE',
  'pc3S': 'abâdOn6rE',
  'pi1P': 'abâdOnô',
  'pi1S': 'abâdOn',
  'pi2P': 'abâdOne',
  'pi2S': 'abâdOn',
  'pi3P': 'abâdOn',
  'pi3S': 'abâdOn',
  'ppFP': 'abâdOne',
  'ppFS': 'abâdOne',
  'ppMP': 'abâdOne',
  'ppMS': 'abâdOne',
  'ps1P': 'abâdOnjô',
  'ps1S': 'abâdOn',
  'ps2P': 'abâdOnje',
  'ps2S': 'abâdOn',
  'ps3P': 'abâdOn',
  'ps3S': 'abâdOn'}}

# Calcul des performances

In [98]:
correct=0
different=0
missing=0
for ix,row in echantillon.iloc[:,:].iterrows():
    # print(row.lexeme)
    dictPlatinum=row.dropna().to_dict()
    del dictPlatinum["lexeme"]
    # print("Echantillon",dictPlatinum)
    selPlatinum=[]
    selResultats=[]
    for k,v in dictPlatinum.items():
        selPlatinum.append("(platinum.%s=='%s')"%(k,v))
        if k in paradigmes.columns:
            selResultats.append("(paradigmes.%s=='%s')"%(k,v))
    exec("%s=%s"%("testPlatinum","&".join(selPlatinum)))
    if selResultats:
        exec("%s=%s"%("testResultats","&".join(selResultats)))
    else:
        testResultats=[]
    lCandidats=platinum.loc[testPlatinum,:]["lexeme"].tolist()
    dictCandidats=gold.loc[gold.lexeme.isin(lCandidats),:].T.dropna().to_dict()
    # print(dictCandidats)
    if not isinstance(testResultats,list):
        lResultats=paradigmes.loc[testResultats,:]["lexeme"].tolist()
    else:
        lResultats=[row.lexeme]
    # print(lResultats)
    lCorrect=0
    lDifferent=0
    lMissing=0
    for ik,candidat in dictCandidats.items():
        for case,forme in candidat.items():
            for lexeme in lResultats:
                if case in paradigmes.columns and case!="lexeme":
                    sForme=paradigmes.loc[paradigmes.lexeme==lexeme,case]
                    # print(sForme)
                    if (sForme==forme).all():
                        # print("correct",case,end=", ")
                        lCorrect+=1
                    elif (sForme==sForme).all():
                        # print("different",case,forme,sForme.tolist(),end=", ")
                        lDifferent+=1
                    else:
                        # print("missing",case,end=", ")
                        lMissing+=1
    # print()
    # print(row.lexeme,lCorrect,lDifferent,lMissing)
    correct+=lCorrect
    different+=lDifferent
    missing+=lMissing
    # print()
correct,different,missing

(32154, 659, 6918)

In [99]:
float(correct)/(correct+different),float(correct)/(correct+missing)

(0.9799164965105294, 0.8229422604422605)